In [0]:
from pyspark.sql.functions import *

In [0]:
%sql
-- BY DEFAULT - DELTA
create table pyspark_scenarios_catalog.source.productsscd2src
as select * from pyspark_scenarios_catalog.source.products

In [0]:
df = spark.read.table('pyspark_scenarios_catalog.source.productsscd2src')

df = df.withColumn('EffectiveStartDate', col('updatedDate')).withColumn('EffectiveEndDate', lit('4712-12-31').cast('timestamp')).withColumn('isActive', lit('Y'))

df.write.saveAsTable('pyspark_scenarios_catalog.destination.productsscd2trg')

In [0]:
# DEDUP

df_src = spark.sql("""
                   select * from (
                   select *, max(updatedDate) over (partition by id) as max_update_date
                   from pyspark_scenarios_catalog.source.productsscd2src
                   ) cc where cc.updatedDate = cc.max_update_date
                   """)
df_src.createOrReplaceTempView('src')

In [0]:
%sql
select * from src

In [0]:
%sql
merge into pyspark_scenarios_catalog.destination.productsscd2trg trg
using src
on trg.id = src.id
and trg.isActive = 'Y'

when matched
--and other conditions
and trg.updatedDate <> src.updatedDate
then update set trg.isActive = 'N',
trg.EffectiveEndDate = src.updatedDate;

merge into pyspark_scenarios_catalog.destination.productsscd2trg trg
using src
on trg.id = src.id
and trg.isActive = 'Y'

when not matched
then insert (id, name, price, category, updatedDate, EffectiveStartDate, EffectiveEndDate, isActive)
values (src.id, src.name, src.price, src.category, src.updatedDate, src.updatedDate, timestamp '4712-12-31', 'Y')

In [0]:
%sql
select * from pyspark_scenarios_catalog.destination.productsscd2trg
order by id, updatedDate

In [0]:
%sql
select * from pyspark_scenarios_catalog.destination.productsscd2trg
where isActive='Y'
order by id, updatedDate

In [0]:
df_trg = spark.sql("""
                   select * from pyspark_scenarios_catalog.destination.productsscd2trg
                   """)
df_trg.write.format('delta').option('inferSchema', 'true').save('/Volumes/pyspark_scenarios_catalog/source/db_volume/scd2_merge_delta_tbl')

In [0]:
# MERGE - will only work with Delta Table
from delta.tables import DeltaTable

dlt_tbl_obj = DeltaTable.forPath(spark, '/Volumes/pyspark_scenarios_catalog/source/db_volume/scd2_merge_delta_tbl/')


dlt_tbl_obj.alias('trg').merge(
    df_src.alias('src'),
    "src.id = trg.id and cast(trg.EffectiveEndDate as date) = to_date('4712-12-31')"
).whenMatchedUpdate(condition="trg.updatedDate != src.updatedDate",
                    set = {"EffectiveEndDate":"src.updatedDate"})\
    .execute()

dlt_tbl_obj.alias('trg').merge(
    df_src.alias('src'),
    "src.id = trg.id and cast(trg.EffectiveEndDate as date) = to_date('4712-12-31')"
).whenNotMatchedInsert(values = {"id":"src.id", "name":"src.name", "price":"src.price", "category":"src.category", "updatedDate":"src.updatedDate", "EffectiveStartDate":"src.updatedDate", "EffectiveEndDate":"to_timestamp('4712-12-31')"})\
    .execute()

In [0]:
%sql
--truncate table delta.`/Volumes/pyspark_scenarios_catalog/source/db_volume/scd2_merge_delta_tbl`
select * from delta.`/Volumes/pyspark_scenarios_catalog/source/db_volume/scd2_merge_delta_tbl`

In [0]:
%sql
select * from delta.`/Volumes/pyspark_scenarios_catalog/source/db_volume/scd2_merge_delta_tbl`

In [0]:
print(1)